# Seasonal Forcing Ensemble Experiment

This notebook implements the full workflow for a **seasonal forcing perturbation experiment**:

1. **Optimize** model parameters (or load existing calibration)
2. **Run** a 20-year baseline simulation with optimized parameters
3. **Build ensembles** by replacing one season of the 21st year with that season from each of the prior 20 years
4. **Run** all ensemble members through SUMMA
5. **Evaluate** sensitivity via water balance (ET, storage change) and runoff ratio
6. **Visualize** with spaghetti plots of precipitation and streamflow signals

---

## 0. Setup & Imports

In [ ]:
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import yaml
import matplotlib.pyplot as plt

# Add project root to path
PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'ess-project' / 'modeling'))

from seasonal_ensemble_experiment import (
    SeasonalEnsembleExperiment,
    ExperimentConfig,
    ParameterOptimizer,
    ModelRunner,
    ForcingEnsembleBuilder,
    EnsembleRunner,
    ParallelEnsembleRunner,
    EnsembleEvaluator,
    EnsembleVisualizer,
    SEASONS,
)

print(f"Project root: {PROJECT_ROOT}")

## 1. Configuration

Set up the experiment by pointing to your CONFLUENCE config file and defining the time windows.

In [ ]:
# ============================================================
# USER CONFIGURATION — Edit these values for your experiment
# ============================================================

# Path to your CONFLUENCE config YAML
CONFIG_PATH = str(PROJECT_ROOT / 'ess-project' / '0_config_files' / 'config_East_River_lumped.yaml')

# 20-year baseline period
BASELINE_START = '1999-01-02 01:00'
BASELINE_END   = '2020-12-31 23:00'

# Target year (the 21st year — the one we perturb)
TARGET_YEAR = 2021

# Donor years — seasons from these years will replace the target year's seasons
DONOR_YEARS = list(range(1999, 2021))  # 20 years

# Skip optimization if results already exist?
SKIP_OPTIMIZATION = False

# Basin area in m² (for unit conversion m/s → m³/s)
BASIN_AREA_M2 = None  # set if needed for m³/s conversion

# Optional: path to observations CSV (auto-detected if None)
OBS_CSV = None

# Parallel execution settings
MAX_WORKERS = 12          # concurrent SUMMA processes
USE_PARALLEL = True      # True=parallel, False=sequential
GENERATE_SCRIPT = True   # Also generate a bash script for nohup overnight runs

print(f"Config: {CONFIG_PATH}")
print(f"Baseline: {BASELINE_START} to {BASELINE_END}")
print(f"Target year: {TARGET_YEAR}")
print(f"Donor years: {DONOR_YEARS[0]} to {DONOR_YEARS[-1]} ({len(DONOR_YEARS)} years)")
print(f"Total ensemble members: {len(DONOR_YEARS) * 4} (4 seasons × {len(DONOR_YEARS)} donors)")
print(f"Parallel workers: {MAX_WORKERS}")

In [ ]:
# Initialize the experiment
exp = SeasonalEnsembleExperiment(CONFIG_PATH, max_workers=MAX_WORKERS)

# Override basin area if specified
if BASIN_AREA_M2 is not None:
    exp.cfg.basin_area_m2 = BASIN_AREA_M2

print(f"Domain: {exp.cfg.domain_name}")
print(f"Project dir: {exp.cfg.project_dir}")
print(f"Ensemble dir: {exp.cfg.ensemble_dir}")
print(f"Plots dir: {exp.cfg.plots_dir}")
print(f"SUMMA settings: {exp.cfg.settings_dir}")
print(f"Forcing input: {exp.cfg.summa_input_dir}")

---
## 1b. Check Data Coverage

Verify that forcing files and observations cover the full experiment period before starting.

In [ ]:
report = exp.check_data_coverage(
    baseline_start_year=int(BASELINE_START[:4]),
    target_year=TARGET_YEAR,
)

if not report['forcing_ok']:
    print(f"\nFORCING GAPS FOUND: {report['forcing_gaps']}")
    print("Run the cell below (Step 0 — Prepare Data) to fetch missing forcing.")
elif not report['obs_ok']:
    print(f"\nObs starts {report['obs_start'].year}, gaps: {report['obs_gap_years']}")
    print("Obs gaps only affect evaluation plots. Run Step 0 below if you want to extend.")
else:
    print("\nAll data present — ready to run experiment.")

---
## 1c. Step 0 — Prepare Data (optional)

Download observations and/or create basin-averaged forcing files via CONFLUENCE's DataManager.  
**Only run this cell if the coverage check above flagged missing data.**  
Requires internet access (e.g. run from a login node, not a compute node).

In [ ]:
# Uncomment and adjust as needed:
# exp.step0_prepare_data(
#     start_date="1981-01-01",
#     end_date="2019-12-31",
#     download_obs=True,        # fetch USGS streamflow observations
#     create_forcing=True,      # download ERA5 + basin-average
# )

---
## 2. Step 1 — Optimize Parameters

Run OSTRICH/DDS to find best parameters. Set `SKIP_OPTIMIZATION = True` above to reuse existing results.

In [ ]:
best_params = exp.step1_optimize(skip_if_exists=SKIP_OPTIMIZATION)

print("\n" + "=" * 50)
print("Best Parameters:")
print("=" * 50)
display(best_params)

---
## 3. Step 2 — Long-Term Baseline Run (20 years)

Run SUMMA for the full 20-year baseline period using the optimized parameters.

In [ ]:
baseline_output = exp.step2_long_term_run(
    start=BASELINE_START,
    end=BASELINE_END,
    experiment_id='longterm_baseline'
)
print(f"\nBaseline output: {baseline_output}")

---
## 3b. Step 2b — Create Warm State from Baseline

Extract the final state of the 20-year baseline run and write a `warmState.nc` file.  
This replaces the default cold state (all zeros) so the target year and ensemble runs
start from a physically realistic, spun-up state.

In [ ]:
warm_state = exp.step2b_create_warm_state()
print(f"\nWarm state file: {warm_state}")
print("fileManager.txt now points to warmState.nc for all subsequent runs.")

---
## 4. Step 3 — Target Year Baseline (Year 21, Unperturbed)

Run the target year with no modifications to establish the control for comparison.

In [ ]:
target_baseline = exp.step3_target_year_baseline(TARGET_YEAR)
print(f"\nTarget year baseline output: {target_baseline}")

---
## 5. Step 4 — Build Seasonal Forcing Ensembles

For each season (DJF, MAM, JJA, SON), create 20 forcing files where that season's forcing
in the target year is replaced by the same season from each of the 20 donor years.

This creates **80 ensemble members** total (20 per season × 4 seasons).

In [ ]:
# Load full forcing data and build ensembles
ensemble_files = exp.step4_build_ensembles(
    target_year=TARGET_YEAR,
    donor_years=DONOR_YEARS
)

print("\nEnsemble files created:")
for season, files in ensemble_files.items():
    print(f"  {season}: {len(files)} members")

---
## 6. Step 5 — Run All Ensemble Members

Two execution modes:

**Mode A — Python-managed parallel** (`USE_PARALLEL=True`):  
Launches up to `MAX_WORKERS` SUMMA processes concurrently, blocks until all finish.

**Mode B — Overnight bash script** (`GENERATE_SCRIPT=True`):  
Generates `run_ensemble.sh` that you launch with `nohup`. Useful for leaving runs overnight.

> **Naming convention:** Each output is filed under  
> `seasonal_ensemble/results/{SEASON}/from_{DONOR_YEAR}/` with prefix  
> `{DOMAIN}_{SEASON}_from{DONOR_YEAR}` (e.g., `East_River_lumped_MAM_from2005_timestep.nc`).

In [ ]:
# --- Mode A: Run in parallel from Python (blocks until done) ---
if USE_PARALLEL:
    ensemble_results = exp.step5_run_ensembles(
        TARGET_YEAR, parallel=True, max_workers=MAX_WORKERS, poll_interval=30.0
    )
else:
    ensemble_results = exp.step5_run_ensembles(TARGET_YEAR, parallel=False)

print("\nEnsemble results:")
for season, results in ensemble_results.items():
    print(f"  {season}: {len(results)} successful runs")

### 6b. Alternative: Generate Overnight Run Script

Run this **instead of** the cell above if you want to launch via `nohup` and check results tomorrow.
After the script finishes, come back and run the "Collect Results" cell below.

In [ ]:
# --- Mode B: Generate bash script for overnight execution ---
# (Only run this if you did NOT run Mode A above)

if GENERATE_SCRIPT:
    # step4 must have run first to create ensemble forcing files
    script_path = exp.step5_generate_script(
        target_year=TARGET_YEAR,
        max_workers=MAX_WORKERS,
    )
    
    log_dir = exp.cfg.ensemble_dir / 'logs'
    print(f"\nScript generated: {script_path}")
    print(f"\nTo run overnight:")
    print(f"  nohup bash {script_path} > {log_dir}/ensemble_main.log 2>&1 &")
    print(f"\nTo monitor progress:")
    print(f"  tail -f {log_dir}/ensemble_main.log")

### 6c. Collect Results After Overnight Run

Run this cell the morning after launching the script. It scans the results
directory for completed output files so you can proceed to evaluation.

In [ ]:
# --- Collect results from a completed overnight run ---
# (Only run this if you used Mode B / nohup script)

ensemble_results = exp.step5_collect_results()

print("Collected results:")
for season, results in ensemble_results.items():
    print(f"  {season}: {len(results)} completed members")

---
## 7. Step 6 — Evaluate Sensitivity

Extract water balance components from each ensemble member and compute:
- **ET** (total evapotranspiration)
- **Storage change** (ΔS = ΔSWE + Δsoil water)
- **Runoff ratio** (Q/P)
- **Precipitation–runoff elasticity**

In [ ]:
eval_results, sensitivity_summary = exp.step6_evaluate()

print("\n" + "=" * 60)
print("EVALUATION RESULTS (per ensemble member)")
print("=" * 60)
display(eval_results.head(10))

In [ ]:
print("\n" + "=" * 60)
print("SENSITIVITY SUMMARY (per season)")
print("=" * 60)
display(sensitivity_summary)

### Interpretation Guide

| Metric | Description | High Value Means |
|--------|------------|------------------|
| `Q_std` | Std dev of seasonal runoff across ensemble | High sensitivity of runoff to that season's forcing |
| `ET_std` | Std dev of seasonal ET | ET is sensitive to interannual forcing variability |
| `delta_S_std` | Std dev of storage change | Storage response varies with forcing |
| `RR_std` | Std dev of runoff ratio | Partitioning is sensitive to forcing |
| `elasticity_Q_to_P` | Slope of %ΔQ vs %ΔP | >1 = amplified response, <1 = buffered |

---
## 8. Step 7 — Visualization

### 8a. Spaghetti Plots (Individual Seasons)

Each plot shows:
- **Top panel**: Precipitation time series — baseline (black) + ensemble members (colored)
- **Bottom panel**: Streamflow response — baseline (black) + ensemble members (colored) + observations (red dashed)

In [ ]:
plot_paths = exp.step7_visualize(target_year=TARGET_YEAR, obs_csv=OBS_CSV)

print("\nGenerated plots:")
for p in plot_paths:
    print(f"  {p}")

### 8b. Display Individual Season Spaghetti Plots

In [ ]:
from IPython.display import Image, display as ipy_display

for season in SEASONS:
    plot_file = exp.cfg.plots_dir / f'spaghetti_{season}_{TARGET_YEAR}.png'
    if plot_file.exists():
        print(f"\n{'=' * 50}")
        print(f"{season} ({SEASONS[season]['name']})")
        print(f"{'=' * 50}")
        ipy_display(Image(filename=str(plot_file), width=900))

### 8c. Combined 4-Season Spaghetti Plot

In [ ]:
combined_plot = exp.cfg.plots_dir / f'combined_spaghetti_{TARGET_YEAR}.png'
if combined_plot.exists():
    ipy_display(Image(filename=str(combined_plot), width=1000))

### 8d. Sensitivity Dashboard

In [ ]:
dashboard_plot = exp.cfg.plots_dir / 'sensitivity_dashboard.png'
if dashboard_plot.exists():
    ipy_display(Image(filename=str(dashboard_plot), width=1000))

---
## 9. Detailed Analysis

### 9a. Water Balance Comparison by Season

In [ ]:
# Reload results if needed
results_path = exp.cfg.ensemble_dir / 'ensemble_evaluation_results.csv'
if results_path.exists():
    eval_df = pd.read_csv(results_path)
else:
    eval_df = exp.eval_results

# Water balance per season
fig, axes = plt.subplots(1, 4, figsize=(20, 5), sharey=True)

for i, season in enumerate(SEASONS):
    ax = axes[i]
    sdf = eval_df[eval_df['season'] == season]
    
    if sdf.empty:
        ax.set_title(f'{season} (no data)')
        continue
    
    # Stacked bar of P, ET, Q, dS for each member
    x = range(len(sdf))
    ax.bar(x, sdf['P_total'], alpha=0.7, label='P', color='#2166ac')
    ax.bar(x, -sdf['ET_total'], alpha=0.7, label='ET', color='#4dac26')
    ax.bar(x, -sdf['Q_total'], alpha=0.7, label='Q', color='#d01c8b', bottom=-sdf['ET_total'])
    
    ax.set_title(f"{season} ({SEASONS[season]['name']})", fontweight='bold',
                color=SEASONS[season]['color'])
    ax.set_xlabel('Ensemble Member')
    ax.axhline(0, color='black', linewidth=0.5)
    ax.grid(True, alpha=0.3, axis='y')
    
    if i == 0:
        ax.set_ylabel('Water Flux (m/s)')
        ax.legend()

plt.suptitle('Water Balance Components by Season', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 9b. Runoff Ratio Sensitivity

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Runoff ratio distribution
ax = axes[0]
for season in SEASONS:
    sdf = eval_df[eval_df['season'] == season]
    if not sdf.empty:
        ax.hist(sdf['runoff_ratio'].dropna(), bins=10, alpha=0.5,
               label=season, color=SEASONS[season]['color'])

ax.set_xlabel('Runoff Ratio (Q/P)')
ax.set_ylabel('Count')
ax.set_title('Runoff Ratio Distribution', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Right: P-Q elasticity
ax = axes[1]
for season in SEASONS:
    sdf = eval_df[eval_df['season'] == season]
    if not sdf.empty:
        ax.scatter(sdf['P_anomaly_pct'], sdf['Q_anomaly_pct'],
                  color=SEASONS[season]['color'], label=season,
                  alpha=0.7, s=60, edgecolors='white', linewidth=0.5)

lims = ax.get_xlim()
ax.plot(lims, lims, 'k--', alpha=0.3, linewidth=1, label='1:1')
ax.set_xlabel('Precipitation Anomaly (%)')
ax.set_ylabel('Runoff Anomaly (%)')
ax.set_title('Precipitation–Runoff Elasticity', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 9c. ET and Storage Change Sensitivity

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: ET anomaly vs P anomaly
ax = axes[0]
for season in SEASONS:
    sdf = eval_df[eval_df['season'] == season]
    if not sdf.empty:
        ax.scatter(sdf['P_anomaly_pct'], sdf['ET_anomaly_pct'],
                  color=SEASONS[season]['color'], label=season,
                  alpha=0.7, s=60, edgecolors='white', linewidth=0.5)

ax.axhline(0, color='black', linewidth=0.5)
ax.axvline(0, color='black', linewidth=0.5)
ax.set_xlabel('Precipitation Anomaly (%)')
ax.set_ylabel('ET Anomaly (%)')
ax.set_title('ET Sensitivity to Forcing', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Right: Storage change by season
ax = axes[1]
season_data = []
season_labels = []
season_colors = []
for season in SEASONS:
    sdf = eval_df[eval_df['season'] == season]['delta_storage_mean'].dropna()
    if not sdf.empty:
        season_data.append(sdf.values)
        season_labels.append(f"{season}\n({SEASONS[season]['name']})")
        season_colors.append(SEASONS[season]['color'])

if season_data:
    bp = ax.boxplot(season_data, labels=season_labels, patch_artist=True)
    for patch, color in zip(bp['boxes'], season_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)

ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax.set_ylabel('Mean Storage Change (m/s)')
ax.set_title('Storage Change Sensitivity', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

---
## 10. Summary Statistics

In [ ]:
summary_path = exp.cfg.ensemble_dir / 'sensitivity_summary.csv'
if summary_path.exists():
    summary = pd.read_csv(summary_path)
else:
    summary = exp.sensitivity_summary

if summary is not None:
    print("\nSensitivity Summary:")
    print("=" * 70)
    
    # Highlight the most sensitive season for each metric
    for metric in ['Q_std', 'ET_std', 'delta_S_std', 'RR_std']:
        if metric in summary.columns:
            max_idx = summary[metric].idxmax()
            max_season = summary.loc[max_idx, 'season']
            max_val = summary.loc[max_idx, metric]
            print(f"  Most sensitive to {metric:12s}: {max_season} ({max_val:.6f})")
    
    print(f"\nPrecip–Runoff Elasticity:")
    for _, row in summary.iterrows():
        elast = row.get('elasticity_Q_to_P', np.nan)
        interp = '(amplified)' if elast > 1 else '(buffered)' if elast < 1 else ''
        print(f"  {row['season']:3s} ({row['season_name']:6s}): {elast:.2f} {interp}")

    display(summary)

---
## Quick Reference

### Output Naming Convention

All ensemble outputs follow a consistent naming scheme:

| Directory | Pattern | Example |
|-----------|---------|---------|
| Forcing | `{domain}_{season}_{target}_from_{donor}.nc` | `East_River_lumped_MAM_2019_from_2005.nc` |
| Results | `seasonal_ensemble/results/{season}/from_{donor}/` | `results/MAM/from_2005/` |
| Output NC | `{domain}_{season}_from{donor}_timestep.nc` | `East_River_lumped_MAM_from2005_timestep.nc` |
| Logs | `seasonal_ensemble/logs/{season}_from{donor}.log` | `logs/MAM_from2005.log` |

### Execution Modes

| Mode | When to Use | How |
|------|------------|-----|
| **Parallel (Python)** | Interactive, ~30 min total | `exp.step5_run_ensembles(parallel=True)` |
| **Overnight script** | Leave running overnight | `exp.step5_generate_script()` then `nohup bash run_ensemble.sh &` |
| **Sequential** | Debugging individual runs | `exp.step5_run_ensembles(parallel=False)` |

### Running Overnight with tmux

```bash
# 1. Create a tmux session
tmux new -s ensemble

# 2. Activate environment
conda activate CONFLUENCE-base

# 3. Run the full workflow
cd /home/dlhogan/projects/forked-repos/CONFLUENCE-uwmtnhydro/ess-project/modeling
python -c "
from seasonal_ensemble_experiment import SeasonalEnsembleExperiment
exp = SeasonalEnsembleExperiment(
    '../0_config_files/config_East_River_lumped.yaml',
    max_workers=4
)
exp.run_full_workflow(
    baseline_start='1999-01-02 01:00',
    baseline_end='2018-12-31 23:00',
    target_year=2019,
    skip_optimization=True,
    parallel=True,
    max_workers=4,
)
"

# 4. Detach: Ctrl+B then D  (session keeps running)
# 5. Reconnect:  tmux attach -t ensemble
# 6. List:       tmux ls
# 7. Kill:       tmux kill-session -t ensemble
```

**Alternative — generate script first, then nohup:**
```bash
tmux new -s ensemble
conda activate CONFLUENCE-base
cd /home/dlhogan/projects/forked-repos/CONFLUENCE-uwmtnhydro/ess-project/modeling

# Build ensembles (fast) then generate bash script
python -c "
from seasonal_ensemble_experiment import SeasonalEnsembleExperiment
exp = SeasonalEnsembleExperiment('../0_config_files/config_East_River_lumped.yaml', max_workers=4)
exp.step1_optimize(skip_if_exists=True)
exp.step4_build_ensembles(target_year=2019, donor_years=list(range(1999, 2019)))
script = exp.step5_generate_script(target_year=2019, max_workers=4)
print(f'Script ready: {script}')
"

# Launch the script with nohup for extra safety
LOGS=/scratch/dlhogan/ess-project-data/domain_East_River_lumped/seasonal_ensemble/logs
nohup bash /scratch/dlhogan/ess-project-data/domain_East_River_lumped/seasonal_ensemble/run_ensemble.sh \
    > $LOGS/ensemble_main.log 2>&1 &
tail -f $LOGS/ensemble_main.log
# Ctrl+B, D to detach
```

### Individual Steps

- `exp.check_data_coverage()` — Verify forcing + obs coverage
- `exp.step0_prepare_data()` — Download obs / create forcing (needs internet)
- `exp.step1_optimize()` — Find best parameters
- `exp.step2_long_term_run()` — 20-year baseline (adds state vars to output)
- `exp.step2b_create_warm_state()` — Extract final state → warmState.nc
- `exp.step3_target_year_baseline()` — Target year control (uses warm state)
- `exp.step4_build_ensembles()` — Create forcing files
- `exp.step5_run_ensembles()` — Python-managed parallel runs
- `exp.step5_generate_script()` — Generate bash script for nohup
- `exp.step5_collect_results()` — Scan for completed outputs
- `exp.step6_evaluate()` — Water balance metrics
- `exp.step7_visualize()` — Spaghetti plots + dashboard